In [1]:
from pathlib import Path
import torch

# Works whether Jupyter starts from repo root or notebooks/
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

CHECKPOINT = (
    ROOT
    / "results"
    / "checkpoints"
    / "vit_mae_v2_large_data"
    / "best.pt"
)

print("Repo root:", ROOT)
print("Checkpoint:", CHECKPOINT)
print("Exists:", CHECKPOINT.exists())

ckpt = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

print("\nBest epoch:", ckpt["epoch"])
print("Best val loss:", ckpt["best_val_loss"])
print(
    "Experiment:",
    ckpt["config"]["experiment"]["name"]
)

Repo root: /Users/parthshringarpure/deepseti
Checkpoint: /Users/parthshringarpure/deepseti/results/checkpoints/vit_mae_v2_large_data/best.pt
Exists: True

Best epoch: 157
Best val loss: 0.36805915236473086
Experiment: vit_mae_v2_large_data


In [2]:
import sys

sys.path.insert(0, str(ROOT / "src"))

from deepseti.models.vit_mae import ViTMAE
from deepseti.utils.device import detect_device

device = detect_device()
model_cfg = ckpt["config"]["model"]

model = ViTMAE(
    frame_shape=tuple(model_cfg["frame_shape"]),
    patch_size=tuple(model_cfg["patch_size"]),
    channels=model_cfg.get("channels", 1),
    embed_dim=model_cfg["embed_dim"],
    encoder_depth=model_cfg["encoder_depth"],
    encoder_heads=model_cfg["encoder_heads"],
    encoder_mlp_dim=model_cfg["encoder_mlp_dim"],
    decoder_dim=model_cfg["decoder_dim"],
    decoder_depth=model_cfg["decoder_depth"],
    decoder_heads=model_cfg["decoder_heads"],
    decoder_mlp_dim=model_cfg["decoder_mlp_dim"],
    mask_ratio=model_cfg["mask_ratio"],
).to(device)

# strict=True means every checkpoint parameter must match the architecture.
load_result = model.load_state_dict(
    ckpt["model_state_dict"],
    strict=True,
)

model.eval()

num_params = sum(p.numel() for p in model.parameters())

print("Device:", device)
print("Parameters:", f"{num_params:,}")
print("Patch size:", model.patch_size)
print("Mask ratio:", model.mask_ratio)
print("Frame shape:", model.frame_shape)
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

assert num_params == 5_665_664
assert not load_result.missing_keys
assert not load_result.unexpected_keys

print("\n✓ V2-large checkpoint loaded correctly.")

Device: mps
Parameters: 5,665,664
Patch size: (4, 32)
Mask ratio: 0.6
Frame shape: (16, 4096)
Missing keys: []
Unexpected keys: []

✓ V2-large checkpoint loaded correctly.


/Users/parthshringarpure/deepseti/src/deepseti/models/vit_mae.py:429: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
/Users/parthshringarpure/deepseti/src/deepseti/models/vit_mae.py:519: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.decoder = nn.TransformerEncoder(


In [3]:
import json
import numpy as np

BENCH_DIR = ROOT / "data" / "synthetic" / "validation"

clean_frames = np.load(BENCH_DIR / "clean_frames.npy")
injected_frames = np.load(BENCH_DIR / "injected_frames.npy")

with open(BENCH_DIR / "metadata.json") as f:
    metadata = json.load(f)

print("Benchmark:", BENCH_DIR)
print("Clean frames:", clean_frames.shape)
print("Injected frames:", injected_frames.shape)
print("Pairs:", len(metadata))

assert len(clean_frames) == len(injected_frames) == len(metadata)

print("\n✓ Locked development benchmark loaded.")

Benchmark: /Users/parthshringarpure/deepseti/data/synthetic/validation
Clean frames: (700, 16, 4096)
Injected frames: (700, 16, 4096)
Pairs: 700

✓ Locked development benchmark loaded.


In [4]:
import sys

sys.path.insert(0, str(ROOT / "scripts"))

from evaluate_vit_mae import (
    score_paired_dataset_topk,
    compute_metrics,
)

# Geometry comes from the trained model/config — nothing hardcoded from V1.
frame_shape = tuple(model_cfg["frame_shape"])
patch_size = tuple(model_cfg["patch_size"])
mask_ratio = model_cfg["mask_ratio"]

num_time_patches = frame_shape[0] // patch_size[0]
num_freq_patches = frame_shape[1] // patch_size[1]
num_patches = num_time_patches * num_freq_patches

# LOCKED development scoring settings.
TOP_FRACTION = 0.002
BATCH_SIZE = 16
BASE_SEED = 42

print("Patch grid:", num_time_patches, "x", num_freq_patches)
print("Number of patches:", num_patches)
print("Mask ratio:", mask_ratio)
print("Top fraction:", TOP_FRACTION)
print("Batch size:", BATCH_SIZE)
print("Base seed:", BASE_SEED)

clean_scores, injected_scores = score_paired_dataset_topk(
    model=model,
    clean_frames=clean_frames,
    injected_frames=injected_frames,
    num_patches=num_patches,
    mask_ratio=mask_ratio,
    top_fraction=TOP_FRACTION,
    batch_size=BATCH_SIZE,
    base_seed=BASE_SEED,
    device=device,
)

overall = compute_metrics(
    clean_scores,
    injected_scores,
)

print("\n=== V2-LARGE DEVELOPMENT RESULTS ===")
print(f"AUROC           : {overall['auroc']:.6f}")
print(f"AUPRC           : {overall['auprc']:.6f}")
print(f"Paired win rate : {overall['paired_win_rate']:.4f}")

print("\nMean clean score   :", clean_scores.mean())
print("Mean injected score:", injected_scores.mean())

Patch grid: 4 x 128
Number of patches: 512
Mask ratio: 0.6
Top fraction: 0.002
Batch size: 16
Base seed: 42

=== V2-LARGE DEVELOPMENT RESULTS ===
AUROC           : 0.615537
AUPRC           : 0.568190
Paired win rate : 0.8557

Mean clean score   : 135.84373
Mean injected score: 144.86226


In [5]:
import json
from pathlib import Path

RESULT_PATH = ROOT / "results" / "metrics" / "vit_mae_v2_large_data.json"
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)

result = {
    "experiment": "vit_mae_v2_large_data",
    "checkpoint_epoch": int(ckpt["epoch"]),
    "best_val_loss": float(ckpt["best_val_loss"]),
    "train_frames": 4080,
    "validation_frames": 680,
    "benchmark": "data/synthetic/validation",
    "benchmark_pairs": len(metadata),
    "scoring": {
        "method": "masked_topk",
        "top_fraction": TOP_FRACTION,
        "num_inference_masks": 1,
        "batch_size": BATCH_SIZE,
        "base_seed": BASE_SEED,
    },
    "metrics": {
        "auroc": float(overall["auroc"]),
        "auprc": float(overall["auprc"]),
        "paired_win_rate": float(overall["paired_win_rate"]),
        "mean_clean_score": float(clean_scores.mean()),
        "mean_injected_score": float(injected_scores.mean()),
    },
}

with open(RESULT_PATH, "w") as f:
    json.dump(result, f, indent=2)

print("Saved:", RESULT_PATH)
print(json.dumps(result, indent=2))

Saved: /Users/parthshringarpure/deepseti/results/metrics/vit_mae_v2_large_data.json
{
  "experiment": "vit_mae_v2_large_data",
  "checkpoint_epoch": 157,
  "best_val_loss": 0.36805915236473086,
  "train_frames": 4080,
  "validation_frames": 680,
  "benchmark": "data/synthetic/validation",
  "benchmark_pairs": 700,
  "scoring": {
    "method": "masked_topk",
    "top_fraction": 0.002,
    "num_inference_masks": 1,
    "batch_size": 16,
    "base_seed": 42
  },
  "metrics": {
    "auroc": 0.6155367346938776,
    "auprc": 0.5681897799538704,
    "paired_win_rate": 0.8557142857142858,
    "mean_clean_score": 135.84373474121094,
    "mean_injected_score": 144.8622589111328
  }
}
